# Upgrade Runtime
This cell ensures the runtime supports `asyncio` async/await, and is needed on Google Colab. If the runtime is upgraded, you will be prompted to restart it, which you should do before continuing execution.

In [ ]:
%pip install ipython ipykernel --upgrade

# Set Up Environment

### Import required libraries

In [ ]:
%pip install -q canopy

In [ ]:
import canopy
import logging
import nest_asyncio
import json
from pathlib import Path

logging.basicConfig(level=logging.INFO)
nest_asyncio.apply()

### Authenticate

In [4]:
authentication_data = canopy.prompt_for_authentication()
session = canopy.Session(authentication_data)
session.authentication.authenticate()

## Load a local .json file and save it as a user car config

`save_car_config_from_json(filepath)` uses the authenticated `session` above and returns the saved config ID. The JSON export wraps the car data in `config`; `simVersion` and `customProperties` are passed separately. Each function call creates a new car config named after the file stem.

In [5]:
async def save_car_config_from_json(filepath: str | Path) -> str:
    filepath = Path(filepath)

    with filepath.open(encoding="utf-8") as file:
        exported_config = json.load(file)

    config_id = await canopy.create_config(
        session,
        config_type="car",
        name=filepath.stem,
        config_data=exported_config["config"],
        sim_version=exported_config["simVersion"],
        properties=exported_config["customProperties"],
    )

    print(f"Saved car config '{filepath.stem}' with ID: {config_id}")
    return config_id

In [6]:
config_id = await save_car_config_from_json("data\\car\\Canopy GT3.json")

Saved car config 'Canopy GT3' with ID: 88c8bc43cd9b4488879ced9ecf425272


## Load a local .json file and save it as a user car components

`save_car_component_from_json(filepath)` uses the authenticated `session` above and returns the saved component ID. Each call creates a new component named after the file stem, using the subtree `path` from the JSON export. The example below saves a chassis component.


In [7]:
async def save_car_component_from_json(filepath: str | Path) -> str:
    filepath = Path(filepath)

    with filepath.open(encoding="utf-8") as file:
        exported_config = json.load(file)

    config_id = await canopy.create_config(
        session,
        config_type="car",
        name=filepath.stem,
        config_data=exported_config["config"],
        sim_version=exported_config["simVersion"],
        sub_tree_path=exported_config["path"],
    )

    print(f"Saved car.{exported_config['path']} component '{filepath.stem}' with ID: {config_id}")
    return config_id

In [8]:
config_id = await save_car_component_from_json("data\\car_components\\Canopy GT3 Chassis.json")

Saved car.chassis component 'Canopy GT3 Chassis' with ID: 35acd83bbd1c47b4b1c6ee44270fdc1b


## Batch Load all .json files from a local folder and save them as user car components

`batch_save_car_components_from_folder(folder_path)` saves each `.json` file in the folder by calling `save_car_component_from_json` and returns the component IDs in filename order. It raises an error if the folder has no `.json` files. If saving a file fails, the error identifies the file and reports IDs for components saved earlier in the batch; those components are not rolled back. For example: `component_ids = await batch_save_car_components_from_folder(r"data\car_components")`.

In [9]:
async def batch_save_car_components_from_folder(folder_path: str | Path) -> list[str]:
    folder = Path(folder_path)
    if not folder.is_dir():
        raise FileNotFoundError(f"Component folder does not exist: {folder}")

    json_files = sorted(folder.glob("*.json"))
    if not json_files:
        raise FileNotFoundError(f"No .json files found in component folder: {folder}")

    config_ids = []
    for filepath in json_files:
        try:
            config_ids.append(await save_car_component_from_json(filepath))
        except Exception as error:
            saved_ids = ", ".join(config_ids) if config_ids else "none"
            raise RuntimeError(
                f"Failed to save component from '{filepath}'. "
                f"Component IDs already saved in this batch: {saved_ids}."
            ) from error

    return config_ids

In [10]:
config_ids = await batch_save_car_components_from_folder("data\\car_components")

Saved car.aero component 'Canopy GT3 Aero' with ID: 8b848adfbe23403eaf198774693169f1
Saved car.brakes component 'Canopy GT3 Brakes' with ID: 5e863af1e9604461993031fe157b2d7c
Saved car.chassis component 'Canopy GT3 Chassis' with ID: 70b40778a09f42b088ecd694d7864408
Saved car.control component 'Canopy GT3 Control' with ID: 8d0a0dba149b4e8d9c18dfba91421b1e
Saved car.powertrain component 'Canopy GT3 Powertrain' with ID: e33c5bf3111142c08ae0884111e95e11
Saved car.suspension component 'Canopy GT3 Suspension' with ID: ce49f59152b249308698fc0d811f53b9
Saved car.tyres component 'Canopy GT3 Tyres' with ID: 9e92a30058e148b3a886ca9648456680
